In [1]:
from functools import partial
from bo_soaps import soap_worker
import multiprocessing as mp
import warnings
warnings.simplefilter("ignore") # Ignore warnings for now
import argparse

import pandas as pd

import numpy as np
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix, ConfusionMatrixDisplay
from sklearn.gaussian_process.kernels import Matern

#from imblearn.over_sampling import SMOTE

from data import read_molecules

import matplotlib.pyplot as plt

from bayes_opt import BayesianOptimization, acquisition

from bo_soaps import RFModel

In [2]:
df = pd.read_csv("B3DB/B3DB_classification_soaps.csv")
print(df.head())

   NO.                        compound_name  \
0    1                       sulphasalazine   
1    2                           moxalactam   
2    3                           clioquinol   
3    4  bbcpd11 (cimetidine analog) (y-g13)   
4    5                        schembl614298   

                                          IUPAC_name  \
0  2-hydroxy-5-[[4-(pyridin-2-ylsulfamoyl)phenyl]...   
1  7-[[2-carboxy-2-(4-hydroxyphenyl)acetyl]amino]...   
2                       5-chloro-7-iodoquinolin-8-ol   
3  2-[2-[(3-bromopyridin-2-yl)methylsulfanyl]ethy...   
4  (2s,3s,4s,5r)-6-[[(4r,4ar,7s,7ar,12bs)-7-hydro...   

                                              SMILES         CID  logBB  \
0   O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O      5339.0  -2.69   
1  COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...      3889.0  -2.52   
2                             Oc1c(I)cc(Cl)c2cccnc12      2788.0  -2.40   
3                         CCNC(=NCCSCc1ncccc1Br)NC#N  14022517.0  -2.15   
4  CN

In [3]:
df['Mol'] = read_molecules(df, "B3DB/", col='NO.')
print(df.head())

   NO.                        compound_name  \
0    1                       sulphasalazine   
1    2                           moxalactam   
2    3                           clioquinol   
3    4  bbcpd11 (cimetidine analog) (y-g13)   
4    5                        schembl614298   

                                          IUPAC_name  \
0  2-hydroxy-5-[[4-(pyridin-2-ylsulfamoyl)phenyl]...   
1  7-[[2-carboxy-2-(4-hydroxyphenyl)acetyl]amino]...   
2                       5-chloro-7-iodoquinolin-8-ol   
3  2-[2-[(3-bromopyridin-2-yl)methylsulfanyl]ethy...   
4  (2s,3s,4s,5r)-6-[[(4r,4ar,7s,7ar,12bs)-7-hydro...   

                                              SMILES         CID  logBB  \
0   O=C(O)c1cc(N=Nc2ccc(S(=O)(=O)Nc3ccccn3)cc2)ccc1O      5339.0  -2.69   
1  COC1(NC(=O)C(C(=O)O)c2ccc(O)cc2)C(=O)N2C(C(=O)...      3889.0  -2.52   
2                             Oc1c(I)cc(Cl)c2cccnc12      2788.0  -2.40   
3                         CCNC(=NCCSCc1ncccc1Br)NC#N  14022517.0  -2.15   
4  CN

In [4]:
pool = mp.Pool(14, maxtasksperchild=100) # Something keeps leaking memory

In [5]:
def run_bo(centres, neighbours, nu_R=1, nu_S=0, lower=2, upper=10, min_cutoff=5, max_cutoff=20, min_sigma=0.1, max_sigma=2):
    model = RFModel(
        df=df,
        repeats=5,
        nfolds=5,
        Z=centres,
        neighbours=neighbours,
        nu_R=nu_R,
        nu_S=nu_S,
        pool=pool
    )

    pbounds = {
        'cutoff': (min_cutoff, max_cutoff, int),
        'l_max': (lower, upper, int),
        'n_max': (lower, upper, int),
        'sigma': (min_sigma, max_sigma),
    }

    optimiser = BayesianOptimization(
        f=model.fmax,
        acquisition_function=acquisition.ExpectedImprovement(xi=0.01),
        pbounds=pbounds,
        verbose=2,
        random_state=0,
    )

    optimiser.set_gp_params(kernel=Matern(nu=2.5, length_scale=np.ones(len(pbounds))))

    print(f"Running Optimiser\n")
    optimiser.maximize(init_points=10, n_iter=50)

    print(f"Max: {optimiser.max['target']}\n\n")
    return model, optimiser

In [6]:
atoms = [6, 1, 7, 8, 9, 15, 16, 17]

for i in range(len(atoms)):
    a = atoms[:i+1]
    print("Running BayesOpt with atom selection: ", a)
    model, opt = run_bo(
        centres=atoms[:i+1],
        neighbours=atoms[:i+1]
    )
    print("Best parameters for BayesOpt with atom selection: ", a)
    print(opt.max, "\n")
    print("Best scores for BayesOpt with atom selection: ", a)
    print(model.evaluate(**opt.max['params']))
    print("\n\n\n")


Running BayesOpt with atom selection:  [6]
Running Optimiser

|   iter    |  target   |  cutoff   |   l_max   |   n_max   |   sigma   |
-------------------------------------------------------------------------
| 1         | 0.2715597 | 17        | 7         | 2         | 1.2452504 |
| 2         | 0.3699878 | 8         | 9         | 5         | 1.3271988 |
| 3         | 0.4998483 | 9         | 9         | 8         | 0.2077546 |
| 4         | 0.4258230 | 17        | 3         | 8         | 1.6431205 |
| 5         | 0.3521804 | 19        | 10        | 3         | 1.8586336 |
| 6         | 0.3835526 | 18        | 10        | 6         | 0.1384149 |
| 7         | 0.2883235 | 8         | 7         | 2         | 1.9593748 |
| 8         | 0.4180502 | 13        | 3         | 5         | 1.0889072 |
| 9         | 0.2502108 | 8         | 9         | 2         | 1.2058376 |
| 10        | 0.4378994 | 14        | 2         | 6         | 0.9998407 |
| 11        | 0.4733804 | 17        | 10        | 